# 04 - Rede Neural (Keras)
Mesmo protocolo de avaliacao. Usamos Keras com backend PyTorch (como nas aulas).

In [ ]:
# keras rodando com backend torch (como na aula)
import os
os.environ.setdefault("KERAS_BACKEND", "torch")  # nas aulas usamos o backend torch
import keras
print("Keras backend:", keras.backend.backend())

In [ ]:
# bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.options.display.float_format = '{:.3f}'.format
import sys; sys.path.append("../src")   # acesso a preprocessing.py
DATA = "../data/processed"


In [ ]:
# --- pre-processamento feito aqui mesmo, no estilo das aulas ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num = ["idade_anos", "NU_CONTATO", "dias_diag_trat"]
cat = ["CS_SEXO","CS_RACA","CS_ESCOL_N","SG_UF_NOT","RAIOX_TORA","TESTE_TUBE",
       "BACILOSC_E","HIV","AGRAVAIDS","AGRAVALCOO","AGRAVDIABE","AGRAVDOENC",
       "AGRAVDROGA","AGRAVTABAC","TRAT_SUPER","ANT_RETRO","BENEF_GOV",
       "POP_RUA","POP_LIBER","POP_IMIG","POP_SAUDE"]

def preparar(df):
    # cria o atraso ate o inicio do tratamento e deixa as categoricas como texto
    df = df.copy()
    df["DT_DIAG"]    = pd.to_datetime(df["DT_DIAG"], errors="coerce")
    df["DT_INIC_TR"] = pd.to_datetime(df["DT_INIC_TR"], errors="coerce")
    df["dias_diag_trat"] = (df["DT_INIC_TR"] - df["DT_DIAG"]).dt.days
    for c in cat:
        df[c] = df[c].fillna("ignorado").astype(str)
    return df[num + cat]
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, RocCurveDisplay

treino = pd.read_csv(f"{DATA}/treino.csv"); teste1 = pd.read_csv(f"{DATA}/teste1.csv"); teste2 = pd.read_csv(f"{DATA}/teste2.csv")
print("treino:", len(treino), "linhas (base completa)")

def tabela_metricas(y_true, y_pred, y_proba, nome="Modelo"):
    return pd.DataFrame({"acuracia":[accuracy_score(y_true,y_pred)],"recall":[recall_score(y_true,y_pred)],
        "precisao":[precision_score(y_true,y_pred)],"f1":[f1_score(y_true,y_pred)],
        "roc_auc":[roc_auc_score(y_true,y_proba)]}, index=[nome])

### Pre-processamento + balanceamento

In [ ]:
transf = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat),
], verbose_feature_names_out=False)
X_train = np.asarray(transf.fit_transform(preparar(treino)), dtype="float32")
y_train = treino["ltfu"].values
# da mais peso pro abandono, que e a classe minoritaria
peso = {0: 1.0, 1: float((y_train==0).sum()/(y_train==1).sum())}
print("treino:", X_train.shape, "| peso classe abandono =", round(peso[1],2))

### Definicao e treino do modelo

In [ ]:
# rede simples, 2 camadas ocultas, saida sigmoide (probabilidade)
modelo = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.summary()

In [ ]:
history = modelo.fit(X_train, y_train, epochs=10, batch_size=512,
                     validation_split=0.2, class_weight=peso, verbose=1)

In [ ]:
# curva de aprendizado
pd.DataFrame(history.history)[["loss","val_loss"]].plot(figsize=(8,4))
plt.xlabel("Epoca"); plt.title("Curva de aprendizado da rede neural"); plt.grid(True); plt.show()

### Avaliacao no teste1

In [ ]:
X_teste1 = np.asarray(transf.transform(preparar(teste1)), dtype="float32"); y_teste1 = teste1["ltfu"].values
proba1 = modelo.predict(X_teste1, verbose=0).ravel(); pred1 = (proba1 >= 0.5).astype(int)
m1 = tabela_metricas(y_teste1, pred1, proba1, "RedeNeural - Teste1"); m1

In [ ]:
RocCurveDisplay.from_predictions(y_teste1, proba1, plot_chance_level=True)
plt.title("Curva ROC - Rede Neural (Teste1)"); plt.show()

### Rodada 2: junta teste1 ao treino e avalia no teste2

In [ ]:
treino2 = pd.concat([treino, teste1], ignore_index=True)
transf2 = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat),
], verbose_feature_names_out=False)
X_tr2 = np.asarray(transf2.fit_transform(preparar(treino2)), dtype="float32")
y_tr2 = treino2["ltfu"].values
peso2 = {0: 1.0, 1: float((y_tr2==0).sum()/(y_tr2==1).sum())}

modelo2 = keras.Sequential([
    keras.Input(shape=(X_tr2.shape[1],)),
    keras.layers.Dense(32, activation="relu"), keras.layers.Dropout(0.3),
    keras.layers.Dense(16, activation="relu"), keras.layers.Dense(1, activation="sigmoid")])
modelo2.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo2.fit(X_tr2, y_tr2, epochs=10, batch_size=512, validation_split=0.2, class_weight=peso2, verbose=0)

X_teste2 = np.asarray(transf2.transform(preparar(teste2)), dtype="float32"); y_teste2 = teste2["ltfu"].values
proba2 = modelo2.predict(X_teste2, verbose=0).ravel(); pred2 = (proba2 >= 0.5).astype(int)
m2 = tabela_metricas(y_teste2, pred2, proba2, "RedeNeural - Teste2"); m2

In [ ]:
# salvar a rede e os transformadores (para o app)
import joblib
modelo2.save("../models/modelo_rede_neural.keras")
joblib.dump(transf2, "../models/transformadores_nn.joblib")
print("rede neural salva.")